In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from src.data.loader import load_cmapss_raw, add_rul, split_engines
from src.features.engineering import add_rolling_features, fit_scaler, apply_scaler

print("Libraries loaded ✓")

## 1. Load and Label Data

In [ ]:
train_raw, test_raw, rul_series = load_cmapss_raw(data_dir='../data/raw')
train_labeled = add_rul(train_raw, max_rul=125)

print(f"Train engines: {train_labeled['engine_id'].nunique()}")
print(f"RUL range: [{train_labeled['RUL'].min()}, {train_labeled['RUL'].max()}]")
train_labeled[['engine_id', 'cycle', 'RUL']].tail()

## 2. Engine-Based Train / Val / Holdout Split
Splitting by engine ID prevents data leakage between sequences.

In [ ]:
train_df, val_df, holdout_df = split_engines(train_labeled, seed=42)

print(f"Train engines:   {train_df['engine_id'].nunique()} ({len(train_df):,} rows)")
print(f"Val engines:     {val_df['engine_id'].nunique()} ({len(val_df):,} rows)")
print(f"Holdout engines: {holdout_df['engine_id'].nunique()} ({len(holdout_df):,} rows)")

## 3. Rolling Feature Engineering
30-cycle rolling mean and std per sensor captures degradation trends.

In [ ]:
train_feat   = add_rolling_features(train_df)
val_feat     = add_rolling_features(val_df)
holdout_feat = add_rolling_features(holdout_df)
test_feat    = add_rolling_features(test_raw)

feature_cols = [c for c in train_feat.columns if c.endswith('_mean') or c.endswith('_std')]
print(f"Rolling feature count: {len(feature_cols)}")
print(f"Sample features: {feature_cols[:6]}")

## 4. Fit MinMax Scaler on Training Data
One scaler fit on training population, applied to all other sets.

In [ ]:
scaler = fit_scaler(train_feat, model_dir='../models')
train_scaled   = apply_scaler(train_feat,   scaler)
val_scaled     = apply_scaler(val_feat,     scaler)
holdout_scaled = apply_scaler(holdout_feat, scaler)
test_scaled    = apply_scaler(test_feat,    scaler)

print("Scaler saved → models/scaler.pkl")
print(f"Train scaled shape:   {train_scaled.shape}")
print(f"Scaled value range:   [{train_scaled[feature_cols].min().min():.3f}, {train_scaled[feature_cols].max().max():.3f}]")

In [ ]:
# Save processed data
os.makedirs('../data/processed', exist_ok=True)
train_scaled.to_parquet('../data/processed/train.parquet',   index=False)
val_scaled.to_parquet('../data/processed/val.parquet',       index=False)
holdout_scaled.to_parquet('../data/processed/holdout.parquet', index=False)
test_scaled.to_parquet('../data/processed/test.parquet',     index=False)
rul_series.to_frame('RUL').to_parquet('../data/processed/test_rul.parquet', index=False)

print("Saved to data/processed/:")
for f in sorted(os.listdir('../data/processed')):
    size = os.path.getsize(f'../data/processed/{f}') / 1024
    print(f"  {f:30s} {size:.1f} KB")

## Summary
- Train/Val/Holdout split: 80/10/10 engines (no row-level leakage)
- Rolling window: 30 cycles, min_periods=1 (no NaN)
- MinMax scaler fit on training set only → saved to models/scaler.pkl
- Processed data saved to data/processed/ as Parquet files